# H₂ ground-state energy: exact diag vs VQE

Compute the H₂ ground-state energy from kinetic-energy integrals. Same `eigensolve(T)` trace runs exact diag on cpu, VQE on cpu+sim.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** lowest-eigenvalue Hamiltonian. **Classical:** `numpy.linalg.eigh`. **Quantum:** VQE — parameterized ansatz, `expect(H, ψ_θ)` per step, classical optimizer. The kernel (kinetic integrals via Obara-Saika) stays classical; cpusim engagement is at the eigensolve block.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.core import types
from uniqx.domains.chemistry.kernels import kinetic
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

n_basis, n_prim, lmax = 2, 1, 0
exps = np.array([[1.0], [1.0]])
coeffs = np.array([[1.0], [1.0]])
centers = np.array([[0., 0., 0.], [0.74, 0., 0.]])  # H2 bond ≈ 0.74 a.u.
ang = np.array([0, 0], dtype=np.int32)

@trace
def prog(exps, coeffs, centers, ang, state):
    t_flat = kinetic(exps, coeffs, centers, ang, n_basis=n_basis, n_prim=n_prim, lmax=lmax)
    t_mat = shape.reshape(t_flat, result_type=types.tensor("f64", [n_basis, n_basis]),
                           shape=[n_basis, n_basis])
    return apply_linear(t_mat, state)

psi_0 = np.ones(n_basis) / np.sqrt(n_basis)
module = prog(exps.tolist(), coeffs.tolist(), centers.tolist(), ang.tolist(), psi_0.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
